# Data Pre-processing

In [1]:
import sklearn as sk
import pandas as pd

## Import Data

In [23]:
df = pd.read_csv("US_Accidents_March23.csv")
len(df)

7728394

## Basic Data Preprocessing
To simplify your project a bit, use only the data instances that contain at most 1 missing value. According to our counts:
There are 3,554,549 data instances with no missing values:
There are 244,702 data instances with exactly one missing value.
Hence, your dataset for the project should contain 3,554,549 + 244,702 = 3,799,251

In [24]:
df.dropna(thresh=len(df.columns) - 1, inplace=True)
len(df)

3799251

In [25]:
df.drop(columns=['ID', 'Description', 'Country', 'Source', 'Turning_Loop'], inplace = True)
print(df.columns)
basic_df = df.copy(deep=True)

Index(['Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng',
       'End_Lat', 'End_Lng', 'Distance(mi)', 'Street', 'City', 'County',
       'State', 'Zipcode', 'Timezone', 'Airport_Code', 'Weather_Timestamp',
       'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
       'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)',
       'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing',
       'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station',
       'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Sunrise_Sunset',
       'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight'],
      dtype='str')


## Advanced Pre-processing
Goals: 
- Encode all features to numeric values
- Use feature selection/elimination to reduce dimensionality for later analysis

In [26]:
# basic_df is already an independent copy; reuse df to avoid another full-data copy.
advanced_df = df.copy(deep = True)

# Handle datetime -> integer by extracting integer values
advanced_df["Start_Time"] = pd.to_datetime(advanced_df["Start_Time"], format="ISO8601")
advanced_df["End_Time"] = pd.to_datetime(advanced_df["End_Time"], format="ISO8601")

advanced_df["Duration_Minutes"] = (
    advanced_df["End_Time"] - advanced_df["Start_Time"]
).dt.total_seconds() / 60.0
advanced_df["Start_Hour"] = advanced_df["Start_Time"].dt.hour
advanced_df["Start_DayOfWeek"] = advanced_df["Start_Time"].dt.dayofweek  # 0 = Monday, 6 = Sunday
advanced_df["Start_Month"] = advanced_df["Start_Time"].dt.month
advanced_df.drop(columns=["Start_Time", "End_Time"], inplace=True)

In [27]:
# Drop location cols other than lat/long
advanced_df.drop(
    columns=[
        "City", "County", "Zipcode", "Street", "Airport_Code", "State"
    ],
    inplace=True,
)

In [28]:
#Convert sunset measures
sunset_fields = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
for f in sunset_fields:
    advanced_df[f] = advanced_df[f].map({"Day": 1, "Night": 0}) # day = 1, night = 0

In [29]:
# One-hot encode wind direction and weather condition as dense columns
weather_columns = ["Wind_Direction", "Weather_Condition"]
advanced_df = pd.get_dummies(
    advanced_df,
    columns=weather_columns,
    dummy_na=True,
)

In [30]:
#Drop timezone, weather
advanced_df.drop(columns = ["Timezone", "Weather_Timestamp"], inplace = True)

In [31]:
# Fill missing numeric values one column at a time to avoid allocating another full-size frame.
for column in advanced_df.columns:
    missing = advanced_df[column].isna()
    if missing.any():
        fill_value = advanced_df[column].median()
        if pd.isna(fill_value):
            fill_value = 0
        advanced_df[column] = advanced_df[column].fillna(fill_value)

In [32]:
advanced_df.info()

<class 'pandas.DataFrame'>
Index: 3799251 entries, 3402762 to 7728393
Columns: 187 entries, Severity to Weather_Condition_nan
dtypes: bool(166), float64(13), int32(3), int64(5)
memory usage: 1.2 GB


In [33]:
advanced_df.head()

,Severity,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),...,Weather_Condition_Thunder and Hail / Windy,Weather_Condition_Thunder in the Vicinity,Weather_Condition_Thunderstorm,Weather_Condition_Thunderstorms and Rain,Weather_Condition_Tornado,Weather_Condition_Widespread Dust,Weather_Condition_Widespread Dust / Windy,Weather_Condition_Wintry Mix,Weather_Condition_Wintry Mix / Windy,Weather_Condition_nan
3402762,3,40.108910,-83.092860,40.112060,-83.031870,3.230,42.1,36.1,58.0,29.76,...,False,False,False,False,False,False,False,False,False,False
3402767,3,39.172393,-84.492792,39.170476,-84.501798,0.500,37.0,29.8,93.0,29.69,...,False,False,False,False,False,False,False,False,False,False
3402768,2,39.063240,-84.032430,39.067310,-84.058510,1.427,35.6,29.2,100.0,29.66,...,False,False,False,False,False,False,False,False,False,False
3402769,2,39.067080,-84.058550,39.063020,-84.032540,1.423,35.6,29.2,100.0,29.66,...,False,False,False,False,False,False,False,False,False,False
3402771,2,41.375310,-81.820170,41.367860,-81.821740,0.521,33.1,30.0,92.0,29.63,...,False,False,False,False,False,False,False,False,False,False


# Export

In [34]:
%pip install pyarrow

basic_df.to_parquet(
    "basic_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)

advanced_df.to_parquet(
    "advanced_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
